In [11]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

In [12]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [13]:
print("Shape of data:", df.shape)
print("\nSentiment count:\n", df['sentiment'].value_counts())

Shape of data: (50000, 2)

Sentiment count:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [14]:
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()  # lowercasing
    text = re.sub(r'<.*?>', '', text)  # remove HTML tags
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove punctuation

    words = text.split()  # tokenization
    words = [w for w in words if w not in stop_words]  # remove stopwords
    words = [lemmatizer.lemmatize(w) for w in words]  # lemmatization

    return " ".join(words)

df['cleaned'] = df['review'].apply(clean_text)

df[['review','cleaned']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


,review,cleaned
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [15]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['cleaned'])
y = df['sentiment']

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (50000, 5000)


In [16]:
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['cleaned'])

print("BoW shape:", X_bow.shape)

BoW shape: (50000, 5000)


In [17]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.6 MB/s eta 0:00:00


In [18]:
from gensim.models import Word2Vec

sentences = [text.split() for text in df['cleaned']]

print(sentences[0])

['one', 'reviewer', 'mentioned', 'watching', 'oz', 'episode', 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scene', 'violence', 'set', 'right', 'word', 'go', 'trust', 'show', 'faint', 'hearted', 'timid', 'show', 'pull', 'punch', 'regard', 'drug', 'sex', 'violence', 'hardcore', 'classic', 'use', 'word', 'called', 'oz', 'nickname', 'given', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'focus', 'mainly', 'emerald', 'city', 'experimental', 'section', 'prison', 'cell', 'glass', 'front', 'face', 'inwards', 'privacy', 'high', 'agenda', 'em', 'city', 'home', 'many', 'aryan', 'muslim', 'gangsta', 'latino', 'christian', 'italian', 'irish', 'scuffle', 'death', 'stare', 'dodgy', 'dealing', 'shady', 'agreement', 'never', 'far', 'away', 'would', 'say', 'main', 'appeal', 'show', 'due', 'fact', 'go', 'show', 'dare', 'forget', 'pretty', 'picture', 'painted', 'mainstream', 'audience', 'forget', 'charm', 'forget', 'romance', 'oz', 'me

In [19]:
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2)

print("Word2Vec model trained!")

Word2Vec model trained!


In [20]:
import numpy as np

def avg_word2vec(words, model, size):
    vec = np.zeros(size)
    count = 0

    for word in words:
        if word in model.wv:
            vec += model.wv[word]
            count += 1

    if count != 0:
        vec /= count

    return vec

X_w2v = np.array([avg_word2vec(text.split(), w2v_model, 100) for text in df['cleaned']])

print("Word2Vec shape:", X_w2v.shape)

Word2Vec shape: (50000, 100)


In [21]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [22]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

pred1 = lr.predict(X_test)
print("Logistic Regression done")

Logistic Regression done


In [23]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

pred2 = nb.predict(X_test)
print("Naive Bayes done")

Naive Bayes done


In [24]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

pred3 = dt.predict(X_test)
print("Decision Tree done")


Decision Tree done


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Logistic Regression
acc_lr = accuracy_score(y_test, pred1)
prec_lr = precision_score(y_test, pred1, average='weighted')
rec_lr = recall_score(y_test, pred1, average='weighted')
f1_lr = f1_score(y_test, pred1, average='weighted')

# Naive Bayes
acc_nb = accuracy_score(y_test, pred2)
prec_nb = precision_score(y_test, pred2, average='weighted')
rec_nb = recall_score(y_test, pred2, average='weighted')
f1_nb = f1_score(y_test, pred2, average='weighted')

# Decision Tree
acc_dt = accuracy_score(y_test, pred3)
prec_dt = precision_score(y_test, pred3, average='weighted')
rec_dt = recall_score(y_test, pred3, average='weighted')
f1_dt = f1_score(y_test, pred3, average='weighted')

print("Logistic Regression:", acc_lr, prec_lr, rec_lr, f1_lr)
print("Naive Bayes:", acc_nb, prec_nb, rec_nb, f1_nb)
print("Decision Tree:", acc_dt, prec_dt, rec_dt, f1_dt)

Logistic Regression: 0.8892 0.8894729660381203 0.8892 0.889166957319607
Naive Bayes: 0.855 0.8550405769570216 0.855 0.8549874115326606
Decision Tree: 0.7223 0.7222980560894635 0.7223 0.722298847465602


In [26]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [acc_lr, acc_nb, acc_dt],
    "Precision": [prec_lr, prec_nb, prec_dt],
    "Recall": [rec_lr, rec_nb, rec_dt],
    "F1 Score": [f1_lr, f1_nb, f1_dt]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8892,0.889473,0.8892,0.889167
1,Naive Bayes,0.8550,0.855041,0.8550,0.854987
2,Decision Tree,0.7223,0.722298,0.7223,0.722299


Comparison & Insights

In this project, different NLP preprocessing techniques and feature engineering methods were applied to improve model performance.

🔹 Preprocessing Insights:

Text preprocessing played a crucial role in improving the results. Converting text to lowercase, removing punctuation, stopwords, special characters, and applying lemmatization helped in reducing noise and making the data more meaningful for model training.

🔹 Feature Engineering Comparison:
Bag of Words (BoW): Simple and fast but does not capture the importance of words.
TF-IDF: Performed better as it gives more importance to significant words and reduces the impact of common words.
Word2Vec: Captures semantic meaning of words, but is more complex and computationally expensive.

👉 Overall, TF-IDF gave the best performance for this task.

🔹 Model Comparison:
Logistic Regression: Gave the highest accuracy and performed best overall. It handles high-dimensional text data very well.
Naive Bayes: Performed well and is very fast, but slightly less accurate than Logistic Regression.
Decision Tree: Showed lower performance and tendency to overfit on text data.
🔹 Trade-offs:
Simpler models like Naive Bayes are faster but may sacrifice some accuracy.
Logistic Regression provides a good balance between performance and complexity.
Decision Trees are easy to interpret but not ideal for high-dimensional sparse data like text.
Word2Vec provides deeper understanding but increases computation time.
✅ Conclusion:

TF-IDF combined with Logistic Regression provided the best overall performance. Proper preprocessing and feature selection are key factors in building an effective sentiment analysis model.